In [6]:
# =============================================================================
# linking_gp_reservation_rj.ipynb — FINAL CLEAN VERSION
# =============================================================================

import pandas as pd
import ast
import re
from pathlib import Path

REPO_ROOT        = Path("..")
MC_DIR           = REPO_ROOT / "outputs" / "monte carlo simulation rajasthan"
MERGED_RJ_GP_DIR = REPO_ROOT / "outputs" / "merged_rajasthan_gp"
DATA_DIR         = REPO_ROOT / "data"

# =============================================================================
# CELL 1 — Load all source files
# =============================================================================

# ── Constrained valid MC file ─────────────────────────────────────────────────
mc = pd.read_csv(
    MC_DIR / "rajasthan_mc_nfhs4_constrained_valid.csv",
    dtype={"DHSCLUST": int}
)
print(f"MC (constrained valid): {len(mc)} clusters")

# ── Validated treat_probs ─────────────────────────────────────────────────────
treat_probs = pd.read_csv(
    MERGED_RJ_GP_DIR /
    "nfhs4_rj_valid_mc_treatment_status_probabilities_COMPREHENSIVE_LOOKUP.csv",
    dtype={"DHSCLUST": int}
)
treat_probs = treat_probs[[
    "DHSCLUST", "p_known_treatment_mc", "p_unknown_treatment_mc",
    "treatment_certainty_mc", "primary_gp", "primary_gp_prob",
    "primary_gp_dose", "p_never_mc", "p_once_mc", "p_twice_mc",
    "known_gp_mass", "unmatched_gp_mass", "missing_spatial_mass"
]].copy()
print(f"treat_probs: {len(treat_probs)} clusters")

# ── Primary automated reservation history ─────────────────────────────────────
gp_res = pd.read_csv(
    MERGED_RJ_GP_DIR / "rj_gp_lgd_reservation_history_2005_2010.csv",
    dtype={"gp_lgd_code": str}
)
gp_res["gp_lgd_code"] = (
    gp_res["gp_lgd_code"]
    .str.replace(r"\.0$", "", regex=True).str.strip()
)
print(f"GP reservation history (automated): {len(gp_res)} GPs")

# ── Working file: cluster-level primary GP reservation (incl. fuzzy samiti) ───
working = pd.read_csv(
    MERGED_RJ_GP_DIR /
    "cluster_level_working_reservation_match_original_direct_fuzzy_samiti.csv",
    dtype=str
)
working["DHSCLUST"] = pd.to_numeric(working["DHSCLUST"], errors="coerce").astype("Int64")
working["gp_lgd_code_best"] = (
    working["gp_lgd_code_best"]
    .str.replace(r"\.0$", "", regex=True).str.strip()
)
working_matched = working[working["working_reservation_dose"].notna()].copy()
print(f"Working file: {len(working)} clusters, "
      f"{len(working_matched)} with matched reservation dose")
print(f"  Match source breakdown:")
print(working_matched["working_match_source"].value_counts().to_string())

# ── LGD crosswalk for GP names ────────────────────────────────────────────────
lgd = pd.read_csv(
    DATA_DIR / "All India Village to GP LGD codes.csv",
    encoding="latin-1",
    dtype={"Gram Panchayat LGD Code": str},
    usecols=["Gram Panchayat LGD Code", "Gram Panchayat Name",
             "District Name", "Subdistrict Name"]
)
lgd["Gram Panchayat LGD Code"] = (
    lgd["Gram Panchayat LGD Code"]
    .str.replace(r"\.0$", "", regex=True).str.strip()
)
gp_names = (
    lgd.drop_duplicates(subset=["Gram Panchayat LGD Code"])
    .rename(columns={
        "Gram Panchayat LGD Code": "gp_lgd_code",
        "Gram Panchayat Name":     "gp_name",
        "District Name":           "district",
        "Subdistrict Name":        "subdistrict_samiti"
    })
)
print(f"LGD crosswalk: {len(gp_names)} unique GPs")

# =============================================================================
# CELL 2 — Build combined GP reservation lookup
# =============================================================================

# Source A: automated LGD match
res_a = gp_res[[
    "gp_lgd_code", "reservation_dose",
    "reserved_women_2005", "reserved_women_2010"
]].copy()

# Source B: working file primary GP matches (adds fuzzy samiti coverage)
res_b = (
    working_matched[["gp_lgd_code_best", "working_reservation_dose",
                     "working_reserved_women_2005", "working_reserved_women_2010"]]
    .drop_duplicates(subset=["gp_lgd_code_best"])
    .rename(columns={
        "gp_lgd_code_best":           "gp_lgd_code",
        "working_reservation_dose":   "reservation_dose",
        "working_reserved_women_2005":"reserved_women_2005",
        "working_reserved_women_2010":"reserved_women_2010"
    })
    .copy()
)
res_b["reserved_women_2005"] = pd.to_numeric(
    res_b["reserved_women_2005"], errors="coerce")
res_b["reserved_women_2010"] = pd.to_numeric(
    res_b["reserved_women_2010"], errors="coerce")

# Combine: prefer res_a, supplement with res_b
combined_res = (
    pd.concat([res_a, res_b])
    .drop_duplicates(subset=["gp_lgd_code"], keep="first")
)
print(f"Combined reservation lookup:")
print(f"  Automated (res_a):       {len(res_a)} GPs")
print(f"  Working/fuzzy (res_b):   {len(res_b)} GPs")
print(f"  Combined (deduped):      {len(combined_res)} GPs")
print(f"  Net added by fuzzy:      {len(combined_res) - len(res_a)}")

# =============================================================================
# CELL 3 — Explode constrained MC gp_distribution into long format
# =============================================================================

def parse_gp_dist(val):
    if pd.isna(val): return {}
    if isinstance(val, dict): return val
    try: return ast.literal_eval(val)
    except: return {}

rows = []
for _, row in mc.iterrows():
    gp_dist = parse_gp_dist(row["gp_distribution"])
    total_sims = sum(gp_dist.values())
    if total_sims == 0:
        continue
    for gp_code, count in gp_dist.items():
        rows.append({
            "DHSCLUST":    int(row["DHSCLUST"]),
            "gp_lgd_code": str(gp_code).replace(".0", "").strip(),
            "gp_prob":     float(count) / total_sims
        })

mc_long = pd.DataFrame(rows)
mc_long["DHSCLUST"] = mc_long["DHSCLUST"].astype(int)

print(f"Constrained long-format: {len(mc_long)} rows, "
      f"{mc_long['DHSCLUST'].nunique()} clusters, "
      f"{mc_long['gp_lgd_code'].nunique()} unique GPs")

# Verify cluster 290573 (our validation case)
print(f"\nValidation — cluster 290573:")
v = mc_long[mc_long["DHSCLUST"]==290573]
print(v[["gp_lgd_code","gp_prob"]].to_string(index=False))
print(f"Sum: {v['gp_prob'].sum():.4f}  (should be 1.0000)")

# =============================================================================
# CELL 4 — Merge reservation history and cross-check vs treat_probs
# =============================================================================

cluster_gp_res_long = mc_long.merge(
    combined_res, on="gp_lgd_code", how="left"
)

print(f"After reservation merge:")
print(f"  Total rows:          {len(cluster_gp_res_long)}")
print(f"  Matched rows:        {cluster_gp_res_long['reservation_dose'].notna().sum()}")
print(f"  Unmatched rows:      {cluster_gp_res_long['reservation_dose'].isna().sum()}")

# Cross-check rebuilt p_known vs treat_probs
working_clusters = set(treat_probs["DHSCLUST"].unique())

rebuilt = (
    cluster_gp_res_long[
        cluster_gp_res_long["DHSCLUST"].isin(working_clusters) &
        cluster_gp_res_long["reservation_dose"].notna()
    ]
    .groupby("DHSCLUST", as_index=False)["gp_prob"]
    .sum()
    .rename(columns={"gp_prob": "p_known_rebuilt"})
)

check = treat_probs[["DHSCLUST","p_known_treatment_mc"]].merge(
    rebuilt, on="DHSCLUST", how="left"
)
check["p_known_rebuilt"] = check["p_known_rebuilt"].fillna(0)
check["gap"] = check["p_known_rebuilt"] - check["p_known_treatment_mc"]

print(f"\nCross-check vs treat_probs ({len(check)} clusters):")
print(f"  Mean gap:         {check['gap'].mean():.4f}")
print(f"  Max abs gap:      {check['gap'].abs().max():.4f}")
print(f"  Within 0.01:      {(check['gap'].abs() < 0.01).sum()}")
print(f"  Within 0.05:      {(check['gap'].abs() < 0.05).sum()}")
print(f"  Overcounting >0.05:  {(check['gap'] > 0.05).sum()} clusters")
print(f"  Undercounting >0.05: {(check['gap'] < -0.05).sum()} clusters")

# Save authoritative long-format
cluster_gp_res_long.to_csv(
    MERGED_RJ_GP_DIR / "cluster_gp_res_authoritative_long.csv",
    index=False
)
print(f"\nSaved: cluster_gp_res_authoritative_long.csv")

# =============================================================================
# CELL 5 — Identify unmatched GPs within 552 working clusters
# =============================================================================

isolated_clusters = {290069, 290747, 290771}
analysis_clusters = working_clusters - isolated_clusters

# Unmatched GP rows within analysis clusters
unmatched = cluster_gp_res_long[
    cluster_gp_res_long["DHSCLUST"].isin(analysis_clusters) &
    cluster_gp_res_long["gp_lgd_code"].notna() &
    cluster_gp_res_long["reservation_dose"].isna()
].copy()

# Merge treat_probs p_known_treatment_mc for priority binning
unmatched = unmatched.merge(
    treat_probs[["DHSCLUST", "p_known_treatment_mc"]],
    on="DHSCLUST", how="left"
)

print(f"Unmatched GP rows in analysis clusters: {len(unmatched)}")
print(f"Unique unmatched GPs:                   {unmatched['gp_lgd_code'].nunique()}")

# Aggregate per GP
gp_summary = (
    unmatched
    .groupby("gp_lgd_code", as_index=False)
    .agg(
        total_prob_mass             = ("gp_prob",               "sum"),
        n_clusters                  = ("DHSCLUST",              "nunique"),
        max_known_mass_in_clusters  = ("p_known_treatment_mc",  "max"),
        mean_known_mass_in_clusters = ("p_known_treatment_mc",  "mean"),
        min_known_mass_in_clusters  = ("p_known_treatment_mc",  "min"),
    )
    .sort_values("max_known_mass_in_clusters", ascending=False)
    .reset_index(drop=True)
)

# Bin by max_known_mass_in_clusters (from treat_probs — validated)
def assign_bin(val):
    if val >= 0.999:  return "90-99%+"
    elif val >= 0.90: return "90-99%"
    elif val >= 0.75: return "75-90%"
    elif val >= 0.50: return "50-75%"
    elif val >= 0.25: return "25-50%"
    else:             return "0-25%"

gp_summary["known_mass_bin"] = (
    gp_summary["max_known_mass_in_clusters"].apply(assign_bin)
)

print(f"\nKnown mass bin distribution:")
print(gp_summary["known_mass_bin"].value_counts().sort_index())

# =============================================================================
# CELL 6 — Add GP names and export
# =============================================================================

export_df = (
    gp_summary
    .merge(gp_names, on="gp_lgd_code", how="left")
    [[
        "gp_lgd_code", "gp_name", "district", "subdistrict_samiti",
        "total_prob_mass", "n_clusters", "known_mass_bin",
        "max_known_mass_in_clusters", "mean_known_mass_in_clusters",
        "min_known_mass_in_clusters",
    ]]
    .reset_index(drop=True)
)

export_df["reserved_women_2005"] = ""
export_df["reserved_women_2010"] = ""
export_df["notes"] = ""

print(f"Final export shape: {export_df.shape}")
print(f"\nTop 20 priority GPs:")
print(export_df.head(20)[[
    "gp_lgd_code", "gp_name", "district", "subdistrict_samiti",
    "known_mass_bin", "max_known_mass_in_clusters", "n_clusters"
]].to_string(index=False))

print(f"\nPriority summary:")
for b in ["90-99%+", "90-99%", "75-90%", "50-75%", "25-50%", "0-25%"]:
    n = (export_df["known_mass_bin"] == b).sum()
    print(f"  {b}: {n} GPs")

OUTPUT_PATH = MERGED_RJ_GP_DIR / "all_unmatched_gps_for_manual_review.xlsx"

with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
    export_df.to_excel(writer, index=False, sheet_name="Unmatched GPs")
    ws = writer.sheets["Unmatched GPs"]
    for col in ws.columns:
        max_len = max(len(str(c.value)) if c.value else 0 for c in col)
        ws.column_dimensions[col[0].column_letter].width = min(max_len + 2, 40)

print(f"\nSaved to: {OUTPUT_PATH}")
print(f"Total GPs exported: {len(export_df)}")

MC (constrained valid): 1189 clusters
treat_probs: 552 clusters
GP reservation history (automated): 3752 GPs
Working file: 1146 clusters, 552 with matched reservation dose
  Match source breakdown:
working_match_source
1_original_gp_lgd_history_match               417
3_fuzzy_gpname_exact_or_close_samiti_match    103
2_direct_gpname_dhsdistrict_match              32
LGD crosswalk: 231559 unique GPs
Combined reservation lookup:
  Automated (res_a):       3752 GPs
  Working/fuzzy (res_b):   522 GPs
  Combined (deduped):      3877 GPs
  Net added by fuzzy:      125
Constrained long-format: 9733 rows, 1189 clusters, 6064 unique GPs

Validation — cluster 290573:
gp_lgd_code  gp_prob
      39494 0.958217
      39491 0.041783
Sum: 1.0000  (should be 1.0000)
After reservation merge:
  Total rows:          9733
  Matched rows:        3589
  Unmatched rows:      6144

Cross-check vs treat_probs (552 clusters):
  Mean gap:         0.0473
  Max abs gap:      0.5972
  Within 0.01:      223
  Within

In [4]:
import os
for f in sorted(os.listdir("../data/")):
    if "reserv" in f.lower() or "kks" in f.lower() or "sarpanch" in f.lower():
        print(f)

rajasthan gp reservations


In [5]:
for f in sorted(os.listdir("../outputs/merged_rajasthan_gp/")):
    if "reserv" in f.lower():
        print(f)

cluster_level_working_reservation_match_original_direct_fuzzy_samiti.csv
coverage_decomposition_cluster_village_gp_reservation_debug.csv
direct_gpname_dhsdistrict_reservation_match_review.csv
district_level_cluster_village_gp_reservation_coverage_debug.csv
district_level_reservation_merge_coverage_debug.csv
district_level_working_reservation_match_coverage.csv
fuzzy_gpname_reservation_candidates_review.csv
fuzzy_gpname_reservation_rank1_candidates_review.csv
gp_to_reservation_attrition_diagnostic_cluster_level.csv
gp_to_reservation_attrition_summary_by_district.csv
map_nfhs4_rj_full_mc_gp_support_reservation_history_treatment_certainty.png
map_nfhs4_rj_p_any_reserved_deterministic_debug.png
map_nfhs4_rj_p_known_reservation_deterministic_debug.png
map_nfhs4_rj_p_unknown_reservation_deterministic_debug.png
nfhs4_rj_cluster_reservation_dose_probabilities_debug.csv
nfhs4_rj_mc_clusters_full_gp_support_with_reservation_history.csv
rj_gp_lgd_reservation_conflicts_review.csv
rj_gp_lgd_reserva

In [7]:
# =============================================================================
# CELL 7 — Identify which priority GPs push clusters to fully linked
# =============================================================================

priority_gps = set(
    export_df[export_df["known_mass_bin"].isin(["90-99%+", "90-99%"])]
    ["gp_lgd_code"].astype(str).str.strip()
)
print(f"Priority GPs (90-99%+ and 90-99% bins): {len(priority_gps)}")

# For each analysis cluster, find all unmatched GPs in its radius
cluster_unmatched = (
    cluster_gp_res_long[
        cluster_gp_res_long["DHSCLUST"].isin(analysis_clusters) &
        cluster_gp_res_long["reservation_dose"].isna() &
        cluster_gp_res_long["gp_lgd_code"].notna()
    ]
    .groupby("DHSCLUST")["gp_lgd_code"]
    .apply(set)
    .reset_index()
    .rename(columns={"gp_lgd_code": "unmatched_gps"})
)

# Clusters where ALL unmatched GPs are in our priority set
# i.e. matching the priority GPs would fully link these clusters
cluster_unmatched["n_unmatched"] = cluster_unmatched["unmatched_gps"].apply(len)
cluster_unmatched["n_unmatched_priority"] = cluster_unmatched["unmatched_gps"].apply(
    lambda s: len(s & priority_gps)
)
cluster_unmatched["all_unmatched_are_priority"] = (
    cluster_unmatched["n_unmatched"] == cluster_unmatched["n_unmatched_priority"]
)

would_become_fully_linked = cluster_unmatched[
    cluster_unmatched["all_unmatched_are_priority"]
]["DHSCLUST"].tolist()

print(f"\nClusters that become fully linked if all 81 priority GPs matched:")
print(f"  {len(would_become_fully_linked)}")

# For each priority GP, how many clusters does it push to fully linked?
gp_impact = []
for gp in priority_gps:
    # Clusters where this GP is the ONLY remaining unmatched GP
    solo_clusters = cluster_unmatched[
        cluster_unmatched["unmatched_gps"].apply(lambda s: s == {gp})
    ]["DHSCLUST"].tolist()
    
    # Clusters where this GP is one of multiple unmatched GPs
    # but all others are also priority GPs
    contrib_clusters = cluster_unmatched[
        cluster_unmatched["unmatched_gps"].apply(lambda s: gp in s) &
        cluster_unmatched["all_unmatched_are_priority"]
    ]["DHSCLUST"].tolist()

    gp_impact.append({
        "gp_lgd_code": gp,
        "n_solo_fully_linked":   len(solo_clusters),
        "n_contrib_fully_linked": len(contrib_clusters),
        "solo_clusters": solo_clusters
    })

gp_impact_df = pd.DataFrame(gp_impact).merge(
    export_df[["gp_lgd_code","gp_name","district",
               "subdistrict_samiti","max_known_mass_in_clusters",
               "known_mass_bin"]],
    on="gp_lgd_code", how="left"
).sort_values("n_contrib_fully_linked", ascending=False)

print(f"\nPriority GPs ranked by clusters pushed to fully linked:")
print(gp_impact_df[[
    "gp_lgd_code", "gp_name", "district", "subdistrict_samiti",
    "known_mass_bin", "max_known_mass_in_clusters",
    "n_solo_fully_linked", "n_contrib_fully_linked"
]].to_string(index=False))

# Final lookup list: GPs that solo-push at least one cluster to fully linked
solo_pushers = gp_impact_df[gp_impact_df["n_solo_fully_linked"] > 0].copy()
print(f"\nGPs that are the SOLE remaining unmatched GP in at least one cluster:")
print(f"  {len(solo_pushers)} GPs")
print(solo_pushers[[
    "gp_lgd_code", "gp_name", "district", "subdistrict_samiti",
    "n_solo_fully_linked", "max_known_mass_in_clusters"
]].to_string(index=False))

Priority GPs (90-99%+ and 90-99% bins): 81

Clusters that become fully linked if all 81 priority GPs matched:
  50

Priority GPs ranked by clusters pushed to fully linked:
gp_lgd_code             gp_name      district subdistrict_samiti known_mass_bin  max_known_mass_in_clusters  n_solo_fully_linked  n_contrib_fully_linked
     262667              Sootra         Bundi             Talera         90-99%                    0.924797                    0                       2
      35815              Pithas      Bhilwara             Mandal         90-99%                    0.946000                    2                       2
      36364            Khadipur         Bundi             Talera         90-99%                    0.924797                    0                       2
      39491              Shimla Neem Ka Thana             Khetri         90-99%                    0.958217                    2                       2
      41173            Kuanthal     Rajsamand            Deogar

In [9]:
# =============================================================================
# CELL 8 — Export prioritized manual lookup workbook
# =============================================================================

# Tier 1: sole unmatched GP in at least one cluster (22 GPs, unlock 24 clusters)
tier1 = gp_impact_df[gp_impact_df["n_solo_fully_linked"] > 0].copy()
tier1["lookup_tier"] = "Tier 1 — sole unmatched GP (directly fully links cluster)"

# Tier 2: one of multiple unmatched GPs, but all others are also priority GPs
# (matching all in the group fully links those clusters)
tier2 = gp_impact_df[
    (gp_impact_df["n_solo_fully_linked"] == 0) &
    (gp_impact_df["n_contrib_fully_linked"] > 0)
].copy()
tier2["lookup_tier"] = "Tier 2 — part of a group that fully links clusters"

# Tier 3: in 90-99% bin but doesn't contribute to any fully-linked cluster
tier3 = gp_impact_df[
    gp_impact_df["n_contrib_fully_linked"] == 0
].copy()
tier3["lookup_tier"] = "Tier 3 — high priority but no solo fully-linked cluster"

priority_lookup = pd.concat([tier1, tier2, tier3], ignore_index=True)
priority_lookup = priority_lookup.merge(
    gp_names, on="gp_lgd_code", how="left"
)

# Fix: ensure gp_lgd_code is string in both before merging
priority_lookup["gp_lgd_code"] = priority_lookup["gp_lgd_code"].astype(str).str.strip()
gp_names["gp_lgd_code"] = gp_names["gp_lgd_code"].astype(str).str.strip()

# Check if merge worked
print("Columns after concat:", priority_lookup.columns.tolist())
print("gp_name non-null:", priority_lookup["gp_name"].notna().sum() 
      if "gp_name" in priority_lookup.columns else "column missing")

# If gp_name is missing, re-merge explicitly
if "gp_name" not in priority_lookup.columns:
    priority_lookup = priority_lookup.merge(
        gp_names[["gp_lgd_code","gp_name","district","subdistrict_samiti"]],
        on="gp_lgd_code", how="left"
    )
    print(f"Re-merged gp_names: gp_name non-null = {priority_lookup['gp_name'].notna().sum()}")

priority_lookup["reserved_women_2005"] = ""
priority_lookup["reserved_women_2010"] = ""
priority_lookup["notes"] = ""

priority_lookup = priority_lookup[[
    "lookup_tier", "gp_lgd_code", "gp_name", "district", "subdistrict_samiti",
    "known_mass_bin", "max_known_mass_in_clusters",
    "n_solo_fully_linked", "n_contrib_fully_linked",
    "reserved_women_2005", "reserved_women_2010", "notes"
]].reset_index(drop=True)

print(f"\nTier 1 (sole unmatched):  {len(tier1)} GPs → unlock {tier1['n_solo_fully_linked'].sum()} clusters")
print(f"Tier 2 (group unmatched): {len(tier2)} GPs")
print(f"Tier 3 (high priority):   {len(tier3)} GPs")
print(f"Total in priority lookup: {len(priority_lookup)} GPs")

PRIORITY_PATH = MERGED_RJ_GP_DIR / "priority_gps_for_manual_lookup.xlsx"

with pd.ExcelWriter(PRIORITY_PATH, engine="openpyxl") as writer:
    priority_lookup.to_excel(writer, index=False, sheet_name="Priority GPs")
    ws = writer.sheets["Priority GPs"]
    ws.freeze_panes = "A2"
    for col in ws.columns:
        max_len = max(len(str(c.value)) if c.value else 0 for c in col)
        ws.column_dimensions[col[0].column_letter].width = min(max_len + 2, 45)
    from openpyxl.styles import PatternFill
    tier1_fill = PatternFill("solid", fgColor="C6EFCE")
    tier2_fill = PatternFill("solid", fgColor="FFEB9C")
    tier3_fill = PatternFill("solid", fgColor="DAEEF3")
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
        tier = str(row[0].value)
        fill = (tier1_fill if "Tier 1" in tier
                else tier2_fill if "Tier 2" in tier
                else tier3_fill)
        for cell in row:
            cell.fill = fill

print(f"\nSaved to: {PRIORITY_PATH}")
print(f"\nTier 1 GPs to look up first:")
print(priority_lookup[priority_lookup["lookup_tier"].str.contains("Tier 1")][[
    "gp_lgd_code", "gp_name", "district", "subdistrict_samiti",
    "n_solo_fully_linked", "max_known_mass_in_clusters"
]].to_string(index=False))

# Add lookup columns
priority_lookup["reserved_women_2005"] = ""
priority_lookup["reserved_women_2010"] = ""
priority_lookup["notes"] = ""

# Select and order columns
priority_lookup = priority_lookup[[
    "lookup_tier", "gp_lgd_code", "gp_name", "district", "subdistrict_samiti",
    "known_mass_bin", "max_known_mass_in_clusters",
    "n_solo_fully_linked", "n_contrib_fully_linked",
    "reserved_women_2005", "reserved_women_2010", "notes"
]].reset_index(drop=True)

print(f"Tier 1 (sole unmatched):  {len(tier1)} GPs → unlock {tier1['n_solo_fully_linked'].sum()} clusters")
print(f"Tier 2 (group unmatched): {len(tier2)} GPs")
print(f"Tier 3 (high priority):   {len(tier3)} GPs")
print(f"\nTotal in priority lookup: {len(priority_lookup)} GPs")

# Export
PRIORITY_PATH = MERGED_RJ_GP_DIR / "priority_gps_for_manual_lookup.xlsx"

with pd.ExcelWriter(PRIORITY_PATH, engine="openpyxl") as writer:
    priority_lookup.to_excel(writer, index=False, sheet_name="Priority GPs")
    ws = writer.sheets["Priority GPs"]

    # Freeze header row
    ws.freeze_panes = "A2"

    # Auto-width columns
    for col in ws.columns:
        max_len = max(len(str(c.value)) if c.value else 0 for c in col)
        ws.column_dimensions[col[0].column_letter].width = min(max_len + 2, 45)

    # Color-code tiers
    from openpyxl.styles import PatternFill
    tier1_fill = PatternFill("solid", fgColor="C6EFCE")  # green
    tier2_fill = PatternFill("solid", fgColor="FFEB9C")  # yellow
    tier3_fill = PatternFill("solid", fgColor="DAEEF3")  # blue

    for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
        tier = str(row[0].value)
        fill = (tier1_fill if "Tier 1" in tier
                else tier2_fill if "Tier 2" in tier
                else tier3_fill)
        for cell in row:
            cell.fill = fill

print(f"\nSaved to: {PRIORITY_PATH}")
print(f"\nTier 1 GPs to look up first:")
print(priority_lookup[priority_lookup["lookup_tier"].str.contains("Tier 1")][[
    "gp_lgd_code", "gp_name", "district", "subdistrict_samiti",
    "n_solo_fully_linked", "max_known_mass_in_clusters"
]].to_string(index=False))

Columns after concat: ['gp_lgd_code', 'n_solo_fully_linked', 'n_contrib_fully_linked', 'solo_clusters', 'gp_name_x', 'district_x', 'subdistrict_samiti_x', 'max_known_mass_in_clusters', 'known_mass_bin', 'lookup_tier', 'district_y', 'subdistrict_samiti_y', 'gp_name_y']
gp_name non-null: column missing
Re-merged gp_names: gp_name non-null = 81

Tier 1 (sole unmatched):  22 GPs → unlock 24 clusters
Tier 2 (group unmatched): 59 GPs
Tier 3 (high priority):   0 GPs
Total in priority lookup: 81 GPs

Saved to: ../outputs/merged_rajasthan_gp/priority_gps_for_manual_lookup.xlsx

Tier 1 GPs to look up first:
gp_lgd_code             gp_name      district subdistrict_samiti  n_solo_fully_linked  max_known_mass_in_clusters
      35815              Pithas      Bhilwara             Mandal                    2                    0.946000
      39491              Shimla Neem Ka Thana             Khetri                    2                    0.958217
      33733                Nawa         Ajmer        

In [10]:
# =============================================================================
# CELL 9 — Project cluster counts after matching all 81 priority GPs
# =============================================================================

# Current state from treat_probs
print("Current state:")
print(f"  Total working clusters:        {len(treat_probs)}")
print(f"  Fully linked (>=99.9%):        {(treat_probs['p_known_treatment_mc'] >= 0.999).sum()}")
print(f"  >=75% linked:                  {(treat_probs['p_known_treatment_mc'] >= 0.75).sum()}")

# After matching all 81 priority GPs:
# For each cluster, compute how much unmatched probability mass would remain
# if all 81 priority GPs were matched

all_priority_gps = set(
    export_df[export_df["known_mass_bin"].isin(["90-99%+", "90-99%"])]
    ["gp_lgd_code"].astype(str).str.strip()
)

# Probability mass that would be resolved by matching priority GPs
priority_mass_resolved = (
    cluster_gp_res_long[
        cluster_gp_res_long["DHSCLUST"].isin(analysis_clusters) &
        cluster_gp_res_long["gp_lgd_code"].isin(all_priority_gps) &
        cluster_gp_res_long["reservation_dose"].isna()
    ]
    .groupby("DHSCLUST", as_index=False)["gp_prob"]
    .sum()
    .rename(columns={"gp_prob": "mass_resolved"})
)

# New p_known after matching priority GPs
projected = treat_probs[["DHSCLUST","p_known_treatment_mc"]].merge(
    priority_mass_resolved, on="DHSCLUST", how="left"
)
projected["mass_resolved"] = projected["mass_resolved"].fillna(0)
projected["p_known_after"] = (
    projected["p_known_treatment_mc"] + projected["mass_resolved"]
).clip(upper=1.0)

print(f"\nProjected state after matching all 81 priority GPs:")
print(f"  Fully linked (>=99.9%):        {(projected['p_known_after'] >= 0.999).sum()}")
print(f"  >=90% linked:                  {(projected['p_known_after'] >= 0.90).sum()}")
print(f"  >=75% linked:                  {(projected['p_known_after'] >= 0.75).sum()}")
print(f"  Total working clusters:        {len(projected)}")

print(f"\nNet gains:")
print(f"  New fully linked:              "
      f"+{(projected['p_known_after'] >= 0.999).sum() - (treat_probs['p_known_treatment_mc'] >= 0.999).sum()}")
print(f"  New >=75% linked:              "
      f"+{(projected['p_known_after'] >= 0.75).sum() - (treat_probs['p_known_treatment_mc'] >= 0.75).sum()}")

# Distribution after
bins   = [0, 0.25, 0.50, 0.75, 0.90, 0.999, 1.01]
labels = ["0-25%", "25-50%", "50-75%", "75-90%", "90-99%", "100%"]
projected["bin_after"] = pd.cut(
    projected["p_known_after"], bins=bins, labels=labels, right=False
)
print(f"\nFull distribution after matching 81 priority GPs:")
print(projected["bin_after"].value_counts().sort_index())

# Individual women in each tier (using mean cluster size of 25.6)
print(f"\nEstimated individual observations (mean 25.6 women/cluster):")
for label, thresh in [("Fully linked (100%)", 0.999),
                       (">=75% linked", 0.75),
                       (">=50% linked", 0.50)]:
    n = (projected["p_known_after"] >= thresh).sum()
    print(f"  {label}: {n} clusters ≈ {n*25.6:,.0f} women")

Current state:
  Total working clusters:        552
  Fully linked (>=99.9%):        11
  >=75% linked:                  177

Projected state after matching all 81 priority GPs:
  Fully linked (>=99.9%):        39
  >=90% linked:                  78
  >=75% linked:                  186
  Total working clusters:        552

Net gains:
  New fully linked:              +28
  New >=75% linked:              +9

Full distribution after matching 81 priority GPs:
bin_after
0-25%       7
25-50%    120
50-75%    239
75-90%    108
90-99%     39
100%       39
Name: count, dtype: int64

Estimated individual observations (mean 25.6 women/cluster):
  Fully linked (100%): 39 clusters ≈ 998 women
  >=75% linked: 186 clusters ≈ 4,762 women
  >=50% linked: 425 clusters ≈ 10,880 women


In [13]:
# Load the reservation data
reservations = pd.read_csv(
    DATA_DIR / "rajasthan gp reservations" / "sp_2005_2010_manually_reviewed.csv"
)
print(f"Rows: {len(reservations)}")
print(f"Columns: {reservations.columns.tolist()}")

Rows: 9860
Columns: ['sl_no_2005', 'dist_name_2005', 'samiti_name_2005', 'gp_2005', 'reservation_2005', 'name_2005', 'sex_2005', 'category_2005', 'gp_new_2005', 'dist_name_new_2005', 'samiti_name_new_2005', 'dist_2010_2005', 'key_2005', 'sl_no_2010', 'dist_name_2010', 'samiti_name_2010', 'gp_2010', 'reservation_2010', 'name_2010', 'sex_2010', 'category_2010', 'gp_new_2010', 'dist_name_new_2010', 'samiti_name_new_2010', 'key_2010', 'dist', 'nuke']


In [12]:
import os
for f in sorted(os.listdir("../data/rajasthan gp reservations")):
    print(f)

rj_sarpanch_2005.csv
rj_sarpanch_2010.csv
sp_2005_2010_manually_reviewed.csv


In [14]:
import re

def normalize_name(x):
    if pd.isna(x): return ""
    x = str(x).lower().strip()
    x = re.sub(r"[^a-z0-9\s]", "", x)
    x = re.sub(r"\s+", "", x)
    return x

# Filter out nuked rows
nuke_norm = reservations["nuke"].astype(str).str.strip().str.lower()
res_clean = reservations[~nuke_norm.isin(["1","true","yes","y","drop","nuke"])].copy()
print(f"Rows after nuke filter: {len(res_clean)}")

# Use 2010 names where available, fall back to 2005
res_clean["gp_norm"] = res_clean["gp_new_2010"].fillna(res_clean["gp_new_2005"]).apply(normalize_name)
res_clean["dist_norm"] = res_clean["dist_name_new_2010"].fillna(res_clean["dist_name_new_2005"]).apply(normalize_name)
res_clean["samiti_norm"] = res_clean["samiti_name_new_2010"].fillna(res_clean["samiti_name_new_2005"]).apply(normalize_name)

# Check: how many GP names appear more than once within the same district?
gp_district_counts = (
    res_clean.groupby(["dist_norm", "gp_norm"])
    .size()
    .reset_index(name="n_rows")
)

# GPs with multiple rows within same district
duplicate_gp_district = gp_district_counts[gp_district_counts["n_rows"] > 1]
print(f"\nUnique district-GP combinations: {len(gp_district_counts)}")
print(f"District-GP combinations with >1 row: {len(duplicate_gp_district)}")
print(f"Share with >1 row: {len(duplicate_gp_district)/len(gp_district_counts):.1%}")

# Of those duplicates, how many have conflicting reservation doses?
res_clean["reserved_women_2005"] = res_clean["reservation_2005"].apply(
    lambda x: int(str(x).upper().strip().endswith("W")) if pd.notna(x) else 0
)
res_clean["reserved_women_2010"] = res_clean["reservation_2010"].apply(
    lambda x: int(str(x).upper().strip().endswith("W")) if pd.notna(x) else 0
)
res_clean["reservation_dose_n"] = res_clean["reserved_women_2005"] + res_clean["reserved_women_2010"]
res_clean["reservation_dose"] = res_clean["reservation_dose_n"].map({0:"never",1:"once",2:"twice"})

dose_by_gp_district = (
    res_clean.groupby(["dist_norm","gp_norm"])
    .agg(
        n_rows         = ("gp_norm", "size"),
        n_dose_values  = ("reservation_dose", "nunique"),
        dose_values    = ("reservation_dose", lambda x: " | ".join(sorted(set(x.dropna()))))
    )
    .reset_index()
)

conflicted = dose_by_gp_district[dose_by_gp_district["n_dose_values"] > 1]
unique_dose = dose_by_gp_district[dose_by_gp_district["n_dose_values"] == 1]

print(f"\nDistrict-GP pairs with unique dose (safe to match): {len(unique_dose)}")
print(f"District-GP pairs with conflicting dose (unsafe):    {len(conflicted)}")
print(f"Share conflicted: {len(conflicted)/len(dose_by_gp_district):.1%}")

print(f"\nSample conflicted pairs:")
print(conflicted.head(10).to_string(index=False))

Rows after nuke filter: 9860

Unique district-GP combinations: 8267
District-GP combinations with >1 row: 1065
Share with >1 row: 12.9%

District-GP pairs with unique dose (safe to match): 7739
District-GP pairs with conflicting dose (unsafe):    528
Share conflicted: 6.4%

Sample conflicted pairs:
dist_norm     gp_norm  n_rows  n_dose_values  dose_values
    ajmer bhagwanpura       2              2 never | once
    ajmer      moyana       2              2 once | twice
    ajmer        nand       2              2 once | twice
    ajmer      nandla       2              2 once | twice
    ajmer      sadara       4              2 once | twice
    ajmer      salari       3              2 once | twice
    ajmer       sawar       3              2 never | once
    ajmer    shergarh       2              2 never | once
    alwar      babedi       2              2 never | once
    alwar       barod       2              2 once | twice


In [15]:
# How many of our unmatched GPs can be matched by GP name + district alone?

# Build district-only lookup (unique dose only, no samiti required)
district_only_lookup = dose_by_gp_district[
    dose_by_gp_district["n_dose_values"] == 1
][["dist_norm", "gp_norm", "dose_values"]].copy()
district_only_lookup = district_only_lookup.rename(
    columns={"dose_values": "reservation_dose_district_match"}
)

# Get normalized names for our unmatched GPs
unmatched_gps_df = export_df[["gp_lgd_code","gp_name","district","subdistrict_samiti"]].copy()
unmatched_gps_df["gp_norm"] = unmatched_gps_df["gp_name"].apply(normalize_name)
unmatched_gps_df["dist_norm"] = unmatched_gps_df["district"].apply(normalize_name)

# Apply district aliases from the original pipeline
DISTRICT_ALIASES = {
    "chittaurgarh": "chittorgarh",
    "dhaulpur": "dholpur",
    "jalor": "jalore",
    "jhunjhunun": "jhunjhunu",
    "anupgarh": "ganganagar",
    "balotra": "barmer",
    "beawar": "ajmer",
    "deeg": "bharatpur",
    "didwanakuchaman": "nagaur",
    "dudu": "jaipur",
    "gangapurcity": "sawaimadhopur",
    "jaipurgramin": "jaipur",
    "jodhpurgramin": "jodhpur",
    "kekri": "ajmer",
    "khairthaltijara": "alwar",
    "kotputlibehror": "jaipur",
    "neemkathana": "sikar",
    "phalodi": "jodhpur",
    "salumbar": "udaipur",
    "sanchore": "jalore",
    "shahpura": "bhilwara",
}

unmatched_gps_df["dist_norm_hist"] = unmatched_gps_df["dist_norm"].apply(
    lambda x: DISTRICT_ALIASES.get(x, x)
)

# Try matching
matched_district = unmatched_gps_df.merge(
    district_only_lookup,
    left_on=["gp_norm", "dist_norm_hist"],
    right_on=["gp_norm", "dist_norm"],
    how="left"
)

n_newly_matched = matched_district["reservation_dose_district_match"].notna().sum()
print(f"Unmatched GPs total:                          {len(unmatched_gps_df)}")
print(f"Newly matched by district-only lookup:         {n_newly_matched}")
print(f"Still unmatched after district-only:           {len(unmatched_gps_df) - n_newly_matched}")
print(f"\nDose distribution among newly matched GPs:")
print(matched_district["reservation_dose_district_match"].value_counts(dropna=False))

# Breakdown by known_mass_bin
matched_district = matched_district.merge(
    export_df[["gp_lgd_code","known_mass_bin","max_known_mass_in_clusters"]],
    on="gp_lgd_code", how="left"
)
print(f"\nNewly matched GPs by priority bin:")
print(matched_district[
    matched_district["reservation_dose_district_match"].notna()
]["known_mass_bin"].value_counts().sort_index())

Unmatched GPs total:                          1867
Newly matched by district-only lookup:         81
Still unmatched after district-only:           1786

Dose distribution among newly matched GPs:
reservation_dose_district_match
NaN      1786
once       38
never      29
twice      14
Name: count, dtype: int64

Newly matched GPs by priority bin:
known_mass_bin
0-25%      1
25-50%    18
50-75%    44
75-90%    16
90-99%     2
Name: count, dtype: int64


In [16]:
# Are these 81 newly matched GPs already in combined_res?
already_in_combined = set(combined_res["gp_lgd_code"].astype(str).str.strip())

newly_matched_gps = matched_district[
    matched_district["reservation_dose_district_match"].notna()
]["gp_lgd_code"].astype(str).str.strip().tolist()

already_covered = [g for g in newly_matched_gps if g in already_in_combined]
genuinely_new   = [g for g in newly_matched_gps if g not in already_in_combined]

print(f"Of 81 district-only matches:")
print(f"  Already in combined_res: {len(already_covered)}")
print(f"  Genuinely new:           {len(genuinely_new)}")

# For the genuinely new ones, what's the bin breakdown?
genuinely_new_df = matched_district[
    matched_district["gp_lgd_code"].astype(str).str.strip().isin(genuinely_new) &
    matched_district["reservation_dose_district_match"].notna()
]
print(f"\nGenuinely new by priority bin:")
print(genuinely_new_df["known_mass_bin"].value_counts().sort_index())

# How many clusters would these unlock?
new_mass_resolved = (
    cluster_gp_res_long[
        cluster_gp_res_long["DHSCLUST"].isin(analysis_clusters) &
        cluster_gp_res_long["gp_lgd_code"].isin(genuinely_new) &
        cluster_gp_res_long["reservation_dose"].isna()
    ]
    .groupby("DHSCLUST", as_index=False)["gp_prob"]
    .sum()
    .rename(columns={"gp_prob": "mass_resolved"})
)

projected2 = treat_probs[["DHSCLUST","p_known_treatment_mc"]].merge(
    new_mass_resolved, on="DHSCLUST", how="left"
)
projected2["mass_resolved"] = projected2["mass_resolved"].fillna(0)
projected2["p_known_after"] = (
    projected2["p_known_treatment_mc"] + projected2["mass_resolved"]
).clip(upper=1.0)

print(f"\nProjected impact of adding {len(genuinely_new)} district-only matches:")
print(f"  Fully linked (>=99.9%): "
      f"{(projected2['p_known_after'] >= 0.999).sum()} "
      f"(currently {(treat_probs['p_known_treatment_mc'] >= 0.999).sum()})")
print(f"  >=75% linked:           "
      f"{(projected2['p_known_after'] >= 0.75).sum()} "
      f"(currently {(treat_probs['p_known_treatment_mc'] >= 0.75).sum()})")

Of 81 district-only matches:
  Already in combined_res: 0
  Genuinely new:           81

Genuinely new by priority bin:
known_mass_bin
0-25%      1
25-50%    18
50-75%    44
75-90%    16
90-99%     2
Name: count, dtype: int64

Projected impact of adding 81 district-only matches:
  Fully linked (>=99.9%): 11 (currently 11)
  >=75% linked:           198 (currently 177)


In [17]:
# =============================================================================
# CELL 10 — Add district-only matching as Stage D to combined_res
# and rebuild cluster_gp_res_long + export
# =============================================================================

# Build Stage D: district-only matches (unique dose, not already in combined_res)
stage_d = matched_district[
    matched_district["reservation_dose_district_match"].notna() &
    ~matched_district["gp_lgd_code"].astype(str).str.strip().isin(already_in_combined)
][["gp_lgd_code", "reservation_dose_district_match"]].copy()

stage_d = stage_d.rename(columns={
    "reservation_dose_district_match": "reservation_dose"
})

# Add reserved_women columns from dose
dose_to_women = {
    "never": (0, 0),
    "once":  (0, 1),  # assume reserved in 2010 if once
    "twice": (1, 1)
}
stage_d["reserved_women_2005"] = stage_d["reservation_dose"].map(
    lambda x: dose_to_women.get(x, (None,None))[0]
)
stage_d["reserved_women_2010"] = stage_d["reservation_dose"].map(
    lambda x: dose_to_women.get(x, (None,None))[1]
)

print(f"Stage D (district-only) new GPs: {len(stage_d)}")
print(f"Dose distribution:")
print(stage_d["reservation_dose"].value_counts())

# Add to combined_res
combined_res_extended = pd.concat([
    combined_res,
    stage_d[["gp_lgd_code","reservation_dose",
             "reserved_women_2005","reserved_women_2010"]]
]).drop_duplicates(subset=["gp_lgd_code"], keep="first")

print(f"\nExtended combined_res: {len(combined_res_extended)} GPs "
      f"(was {len(combined_res)})")

# Rebuild cluster_gp_res_long with extended lookup
cluster_gp_res_long_v2 = mc_long.merge(
    combined_res_extended, on="gp_lgd_code", how="left"
)

print(f"\nRebuilt long-format:")
print(f"  Matched rows:   {cluster_gp_res_long_v2['reservation_dose'].notna().sum()}")
print(f"  Unmatched rows: {cluster_gp_res_long_v2['reservation_dose'].isna().sum()}")

# Cross-check
rebuilt_v2 = (
    cluster_gp_res_long_v2[
        cluster_gp_res_long_v2["DHSCLUST"].isin(working_clusters) &
        cluster_gp_res_long_v2["reservation_dose"].notna()
    ]
    .groupby("DHSCLUST", as_index=False)["gp_prob"]
    .sum()
    .rename(columns={"gp_prob": "p_known_rebuilt"})
)

check_v2 = treat_probs[["DHSCLUST","p_known_treatment_mc"]].merge(
    rebuilt_v2, on="DHSCLUST", how="left"
)
check_v2["p_known_rebuilt"] = check_v2["p_known_rebuilt"].fillna(0)
check_v2["gap"] = check_v2["p_known_rebuilt"] - check_v2["p_known_treatment_mc"]

print(f"\nCross-check vs treat_probs:")
print(f"  Mean gap:            {check_v2['gap'].mean():.4f}")
print(f"  Max abs gap:         {check_v2['gap'].abs().max():.4f}")
print(f"  Within 0.01:         {(check_v2['gap'].abs() < 0.01).sum()}")
print(f"  Within 0.05:         {(check_v2['gap'].abs() < 0.05).sum()}")
print(f"  Overcounting >0.05:  {(check_v2['gap'] > 0.05).sum()}")
print(f"  Undercounting >0.05: {(check_v2['gap'] < -0.05).sum()}")

# Project new cluster counts using treat_probs + resolved mass
unmatched_v2 = cluster_gp_res_long_v2[
    cluster_gp_res_long_v2["DHSCLUST"].isin(analysis_clusters) &
    cluster_gp_res_long_v2["gp_lgd_code"].notna() &
    cluster_gp_res_long_v2["reservation_dose"].isna()
].copy()

unmatched_v2 = unmatched_v2.merge(
    treat_probs[["DHSCLUST","p_known_treatment_mc"]],
    on="DHSCLUST", how="left"
)

gp_summary_v2 = (
    unmatched_v2
    .groupby("gp_lgd_code", as_index=False)
    .agg(
        total_prob_mass             = ("gp_prob",              "sum"),
        n_clusters                  = ("DHSCLUST",             "nunique"),
        max_known_mass_in_clusters  = ("p_known_treatment_mc", "max"),
        mean_known_mass_in_clusters = ("p_known_treatment_mc", "mean"),
        min_known_mass_in_clusters  = ("p_known_treatment_mc", "min"),
    )
    .sort_values("max_known_mass_in_clusters", ascending=False)
    .reset_index(drop=True)
)

gp_summary_v2["known_mass_bin"] = (
    gp_summary_v2["max_known_mass_in_clusters"].apply(assign_bin)
)

print(f"\nUpdated unmatched GP bin distribution:")
print(gp_summary_v2["known_mass_bin"].value_counts().sort_index())
print(f"Total unmatched GPs remaining: {len(gp_summary_v2)}")

# Save updated long-format
cluster_gp_res_long_v2.to_csv(
    MERGED_RJ_GP_DIR / "cluster_gp_res_authoritative_long.csv",
    index=False
)
print(f"\nSaved updated cluster_gp_res_authoritative_long.csv")

# Update working variables
cluster_gp_res_long = cluster_gp_res_long_v2
combined_res = combined_res_extended

Stage D (district-only) new GPs: 81
Dose distribution:
reservation_dose
once     38
never    29
twice    14
Name: count, dtype: int64

Extended combined_res: 3958 GPs (was 3877)

Rebuilt long-format:
  Matched rows:   3717
  Unmatched rows: 6016

Cross-check vs treat_probs:
  Mean gap:            0.0604
  Max abs gap:         0.5972
  Within 0.01:         198
  Within 0.05:         343
  Overcounting >0.05:  209
  Undercounting >0.05: 0

Updated unmatched GP bin distribution:
known_mass_bin
0-25%       34
25-50%     542
50-75%     842
75-90%     289
90-99%      78
90-99%+      1
Name: count, dtype: int64
Total unmatched GPs remaining: 1786

Saved updated cluster_gp_res_authoritative_long.csv


In [18]:
# Check: for "once" matched GPs, which year were they actually reserved?
once_gps = stage_d[stage_d["reservation_dose"] == "once"]["gp_lgd_code"].tolist()

# Look up in reservation data to find actual year
once_check = res_clean[
    res_clean["gp_norm"].isin(
        unmatched_gps_df[
            unmatched_gps_df["gp_lgd_code"].isin(once_gps)
        ]["gp_norm"].tolist()
    )
][["gp_norm", "dist_norm", "gp_new_2010", "dist_name_new_2010",
   "reservation_2005", "reservation_2010",
   "reserved_women_2005", "reserved_women_2010"]].head(20)

print("Sample 'once' matched GPs — which year reserved?")
print(once_check.to_string(index=False))

print(f"\nAmong all 'once' rows in reservation data:")
print(f"  Reserved in 2005 only (1,0): "
      f"{((res_clean['reserved_women_2005']==1) & (res_clean['reserved_women_2010']==0)).sum()}")
print(f"  Reserved in 2010 only (0,1): "
      f"{((res_clean['reserved_women_2005']==0) & (res_clean['reserved_women_2010']==1)).sum()}")

Sample 'once' matched GPs — which year reserved?
     gp_norm dist_norm  gp_new_2010 dist_name_new_2010 reservation_2005 reservation_2010  reserved_women_2005  reserved_women_2010
   fatehgarh     ajmer    fatehgarh              AJMER              GEN             OBCW                    0                    1
 manoharpura     ajmer  manoharpura              AJMER            GEN W               SC                    1                    0
       kabra     ajmer        kabra              AJMER              GEN             GENW                    0                    1
  kishanpura     ajmer   kishanpura              AJMER              GEN              OBC                    0                    0
       kabra     ajmer        kabra              AJMER              OBC             GENW                    0                    1
   surajpura     ajmer    surajpura              AJMER            OBC W              GEN                    1                    0
      kadera     ajmer       kader

In [19]:
# =============================================================================
# Fix Stage D — carry actual reserved_women_2005/2010 from reservation data
# =============================================================================

# Rebuild district-only lookup with actual year values
district_only_lookup_full = (
    res_clean.groupby(["dist_norm", "gp_norm"])
    .agg(
        n_dose_values        = ("reservation_dose", "nunique"),
        reservation_dose     = ("reservation_dose", "first"),
        reserved_women_2005  = ("reserved_women_2005", "first"),
        reserved_women_2010  = ("reserved_women_2010", "first")
    )
    .reset_index()
)

# Keep only unique-dose pairs
district_only_lookup_full = district_only_lookup_full[
    district_only_lookup_full["n_dose_values"] == 1
].drop(columns="n_dose_values")

print(f"District-only lookup with year values: {len(district_only_lookup_full)} pairs")

# Re-match unmatched GPs
stage_d_fixed = unmatched_gps_df.merge(
    district_only_lookup_full,
    left_on=["gp_norm", "dist_norm_hist"],
    right_on=["gp_norm", "dist_norm"],
    how="left"
)[["gp_lgd_code", "reservation_dose",
   "reserved_women_2005", "reserved_women_2010"]].copy()

stage_d_fixed = stage_d_fixed[
    stage_d_fixed["reservation_dose"].notna() &
    ~stage_d_fixed["gp_lgd_code"].astype(str).str.strip().isin(already_in_combined)
].drop_duplicates(subset=["gp_lgd_code"])

print(f"Stage D fixed GPs: {len(stage_d_fixed)}")
print(f"\nDose distribution:")
print(stage_d_fixed["reservation_dose"].value_counts())
print(f"\nYear breakdown for 'once':")
once = stage_d_fixed[stage_d_fixed["reservation_dose"]=="once"]
print(f"  2005-only (1,0): {((once['reserved_women_2005']==1) & (once['reserved_women_2010']==0)).sum()}")
print(f"  2010-only (0,1): {((once['reserved_women_2005']==0) & (once['reserved_women_2010']==1)).sum()}")

# Rebuild combined_res with fixed Stage D
combined_res_extended_fixed = (
    pd.concat([combined_res[combined_res["gp_lgd_code"].isin(already_in_combined)],
               stage_d_fixed])
    .drop_duplicates(subset=["gp_lgd_code"], keep="first")
)

print(f"\nFixed combined_res: {len(combined_res_extended_fixed)} GPs")

# Rebuild cluster_gp_res_long
cluster_gp_res_long_v2 = mc_long.merge(
    combined_res_extended_fixed, on="gp_lgd_code", how="left"
)

print(f"\nRebuilt long-format:")
print(f"  Matched rows:   {cluster_gp_res_long_v2['reservation_dose'].notna().sum()}")
print(f"  Unmatched rows: {cluster_gp_res_long_v2['reservation_dose'].isna().sum()}")

# Cross-check
rebuilt_v2 = (
    cluster_gp_res_long_v2[
        cluster_gp_res_long_v2["DHSCLUST"].isin(working_clusters) &
        cluster_gp_res_long_v2["reservation_dose"].notna()
    ]
    .groupby("DHSCLUST", as_index=False)["gp_prob"]
    .sum()
    .rename(columns={"gp_prob": "p_known_rebuilt"})
)

check_v2 = treat_probs[["DHSCLUST","p_known_treatment_mc"]].merge(
    rebuilt_v2, on="DHSCLUST", how="left"
)
check_v2["p_known_rebuilt"] = check_v2["p_known_rebuilt"].fillna(0)
check_v2["gap"] = check_v2["p_known_rebuilt"] - check_v2["p_known_treatment_mc"]

print(f"\nCross-check vs treat_probs:")
print(f"  Mean gap:            {check_v2['gap'].mean():.4f}")
print(f"  Max abs gap:         {check_v2['gap'].abs().max():.4f}")
print(f"  Within 0.01:         {(check_v2['gap'].abs() < 0.01).sum()}")
print(f"  Within 0.05:         {(check_v2['gap'].abs() < 0.05).sum()}")
print(f"  Overcounting >0.05:  {(check_v2['gap'] > 0.05).sum()}")
print(f"  Undercounting >0.05: {(check_v2['gap'] < -0.05).sum()}")

# Update working variables
cluster_gp_res_long = cluster_gp_res_long_v2
combined_res = combined_res_extended_fixed

# Save
cluster_gp_res_long.to_csv(
    MERGED_RJ_GP_DIR / "cluster_gp_res_authoritative_long.csv",
    index=False
)
print(f"\nSaved updated cluster_gp_res_authoritative_long.csv")

District-only lookup with year values: 7739 pairs
Stage D fixed GPs: 81

Dose distribution:
reservation_dose
once     38
never    29
twice    14
Name: count, dtype: int64

Year breakdown for 'once':
  2005-only (1,0): 19
  2010-only (0,1): 19

Fixed combined_res: 3958 GPs

Rebuilt long-format:
  Matched rows:   3717
  Unmatched rows: 6016

Cross-check vs treat_probs:
  Mean gap:            0.0604
  Max abs gap:         0.5972
  Within 0.01:         198
  Within 0.05:         343
  Overcounting >0.05:  209
  Undercounting >0.05: 0

Saved updated cluster_gp_res_authoritative_long.csv
